# 6.2 Camera Thermal-Elastic Drift

In this notebook we take a closer look at the inclusion of the Thermal-Elastic Drift (TED) for a PLATO camera. 

We note that the drift motion from the Differential Kinematic Abbreviation (DKA; which varies across the FOV as a function of the velocity vector of the spacescraft's motion) is already implemented in PlatoSim. In theory the DKA can be as high as 20 arcsec (i.e. 1.3 pixel in CCD focal plane) in 3 months. However, this is the level of DKA one would have in principle with a field of view of 90° (in radius). Although large, the PLATO FOV is no larger than ~28° (in radius) such that the DKA is not going to exceed ~0.8 pixel in 3 months.

The mission requirements the TED induced by the platform is not allowed to exceed 0.4 pixel. In practice according to mission contractor called Prime, this is not going to exceed 1/50 pixel in 3 months. Accordingly, the DKA should be the dominant source of drift. Since the latter is fully predictable this can simplify a lot the current mask update strategy (PLATO-LESIA-PDC-DD-0022, i1.0). Indeed, the current one assumes than the drift is not predictable and thus has to be monitored continuously during the quarter for all stars (and all cameras).

### Setup notebook

In [1]:
# Alow changes to the PlatoSim code outside this notebook
%load_ext autoreload
%autoreload 2

# Configure figure in notebook
%matplotlib notebook

### Imports

In [2]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm
from scipy.interpolate import make_interp_spline
from scipy.ndimage import median_filter
from scipy import constants as c

from platosim.noise import *
from platosim.plot import *

---
## 6.2.1 - Red noise drift model
---


---
## 6.2.1 - Polynomial drift model
---


---
## 6.1.2 - Dynamic drift model (from Prime)
---

The Prime is delivering the coefficients of the rotations matrices transforming the pointing directions in the
PLM (payload module) reference frame, so-called **Quarternions**.

Load Prime TED (sampling = 1 h,  duration = 90 d)

In [3]:
# Load file
filename = inputDir + '/04_rev349_ARC3_EOL_90d_3600.csv'
data = np.loadtxt(filename, delimiter=';')

# Time points and sampling
time_ted = data[:,0]                  # [s]
samp_ted = time_ted[1] - time_ted[0]  # [s]

# Prepare for plotting
color = cm.rainbow(np.linspace(0,1,49))
cams = ['CAM 1.1', 'CAM 1.2', 'CAM 1.3', 'CAM 1.4', 'CAM 1.5', 'CAM 1.6', 
        'CAM 2.1', 'CAM 2.2', 'CAM 2.3', 'CAM 2.4', 'CAM 2.5', 'CAM 2.6',
        'CAM 3.1', 'CAM 3.2', 'CAM 3.3', 'CAM 3.4', 'CAM 3.5', 'CAM 3.6',
        'CAM 4.1', 'CAM 4.2', 'CAM 4.3', 'CAM 4.4', 'CAM 4.5', 'CAM 4.6',]

NameError: name 'inputDir' is not defined

### Plot TED time series (cartesian + rotational)

In [ ]:
# Prepare a mean calculation
ted_dir = np.zeros((24,len(data)))
ted_rot = np.zeros((24,len(data)))

fig, ax = plt.subplots(2, 1, figsize=(11,11))

for i,c in zip(range(data.shape[1]-1), color):
    
    # Plot even number i.e. directional in file
    if i % 2 == 0:
        ax[0].plot(time_ted/86400., data[:,i+1], '-', c=c, label='{}'.format(cams[int(i/2)]))
        ted_dir[int(i/2),:] = data[:,i+1]
        
    # Plot odd number i.e. rotational in file
    if i % 2 != 0:
        ax[1].plot(time_ted/86400., data[:,i+1], '-', c=c, label='{}'.format(cams[int(i/2)]))
        ted_rot[int(i/2),:] = data[:,i+1]
        
# Find mean values
ted_dir_mean = np.mean(ted_dir, axis=0)
ted_rot_mean = np.mean(ted_rot, axis=0)          

# Plot mean values
ax[0].plot(time_ted/86400., ted_dir_mean, 'k-', lw=5, label='Mean')  
ax[1].plot(time_ted/86400., ted_rot_mean, 'k-', lw=5, label='Mean')  

# Plot settings
ax[0].set_title('Cartesian TED')   
ax[1].set_title('Rotational TED')   
ax[1].set_xlabel('Time [days]')
ax[0].set_ylabel('Amplitude [arcsec]')
ax[1].set_ylabel('Amplitude [arcsec]')
ax[0].legend(bbox_to_anchor=(1.01, 0.8))
plt.show()

# Save figure
#fig.savefig('/home/nicholas/Nextcloud/presentations/presentation_PW12/plotPrimeTED.png', bbox_inches='tight', dpi=300)

In [ ]:

a = np.array([time_ted, ted_dir[0], ted_dir[0], ted_rot[0]])
print(a.T)
np.savetxt('/home/nicholas/software/python/platonium/models/drift_Ncam1.1_Q1.txt', a.T)

Plot TED PSD

In [ ]:
# First subtract the mean

_, PSD_dir_mean = powerDensityFFT(ted_dir_mean-np.mean(ted_dir_mean)+1, samp_ted)
_, PSD_rot_mean = powerDensityFFT(ted_rot_mean-np.mean(ted_rot_mean)+1, samp_ted)

# Prepare a mean calculation

import math

PSD_dir  = []
PSD_rot  = []
freq_ted = []

for i in range(24):
    
    ted_dir_i = ted_dir[i] - np.mean(ted_dir[i])
    ted_rot_i = ted_rot[i] - np.mean(ted_rot[i])

    freq_ted_i, PSD_dir_i = powerDensityFFT(ted_dir_i, samp_ted)
    _,          PSD_rot_i = powerDensityFFT(ted_rot_i, samp_ted)
    
    freq_ted.append(freq_ted_i)
    PSD_dir.append(PSD_dir_i)
    PSD_rot.append(PSD_rot_i)
   
# Plot Directional TED
fig = plt.figure(figsize=(10,5))
plotPSD(fig, freq_ted, PSD_dir, carbox=False, linewidth=2, xlim=[1e-7, np.max(freq_ted)], ylim=[1e-6, 1e4], 
        units=['Hz', 'arcsec'], title='PSD TED directional', misreq=True)
plt.show()

# PLot Rotational TED
fig = plt.figure(figsize=(10,5))
plotPSD(fig, freq_ted, PSD_rot, carbox=False, linewidth=2, xlim=[1e-7, np.max(freq_ted)], ylim=[1e-6, 1e4], 
        units=['Hz', 'arcsec'], title='PSD TED directional', misreq=True)
plt.show()

# Plot Mean TED
fig = plt.figure(figsize=(10,5))
plotPSD(fig, freq_ted, [PSD_dir_mean, PSD_rot_mean], carbox=False, linewidth=2, xlim=[1e-7, np.max(freq_ted)], ylim=[1e-6, 1e4], 
        units=['Hz', 'arcsec'], title='PSD TED directional', misreq=True)
plt.show()

In [4]:
# Interpolate (piecewise cubic) into higher resolution grid
grid_no  = int(time_ted[-1]/25.)
time_int = np.linspace(time_ted[0], time_ted[-1], grid_no)
samp_int = 25.

# All 24 cams
ted_int_dir = np.zeros((24, grid_no))
ted_int_rot = np.zeros((24, grid_no))

for i in range(24):
    grid_dir = make_interp_spline(time_ted, ted_dir[i], k=3)
    grid_rot = make_interp_spline(time_ted, ted_rot[i], k=3)
    ted_int_dir[i,:] = grid_dir(time_int)
    ted_int_rot[i,:] = grid_rot(time_int)

# Mean values
ted_int_dir_mean = np.mean(ted_int_dir, axis=0)
ted_int_rot_mean = np.mean(ted_int_rot, axis=0)
    
# Find freq and PSD
freq_ted_int, PSD_int_dir_mean = powerDensityFFT(ted_int_dir_mean, samp_int)

freq_jitter, PSD_xJitter = powerDensityFFT(data_jitter_prime[0], samp_jitter)
_,           PSD_yJitter = powerDensityFFT(data_jitter_prime[1], samp_jitter)
_,           PSD_zJitter = powerDensityFFT(data_jitter_prime[2], samp_jitter)

freq = [freq_ted[20], freq_jitter]
PSDx  = [PSD_dir[20], PSD_xJitter]
PSDy  = [PSD_dir[20], PSD_yJitter]
PSDz  = [PSD_rot[20], PSD_zJitter]

# Plot new interpolation for PSD
fig0 = plt.figure(figsize=(9,3.5))
plotPSD(fig0, freq, PSDx, carbox=100, linewidth=1, colors=['c', 'lightgreen'], units=['Hz', 'arcsec'], 
        xlim=[1e-6, np.max(freq_jitter)], ylim=[1e-12, 1e3], misreq=True,
        labels=['TED Dir', 'AOCS Yaw', 'Jitter Yaw'], title='PSD TED directional')
plt.show()

fig1 = plt.figure(figsize=(9,3.5))
plotPSD(fig1, freq, PSDy, carbox=100, linewidth=1, colors=['c', 'limegreen'], units=['Hz', 'arcsec'], 
        xlim=[1e-6, np.max(freq_jitter)], ylim=[1e-12, 1e3], misreq=True,
        labels=['TED Dir', 'AOCS Pitch', 'Jitter Yaw'], title='PSD TED directional')
plt.show()

fig2 = plt.figure(figsize=(9,3.5))
plotPSD(fig2, freq, PSDz, carbox=100, linewidth=1, colors=['b', 'g'], units=['Hz', 'arcsec'], 
        xlim=[1e-6, np.max(freq_jitter)], ylim=[1e-12, 1e3], misreq=True,
        labels=['TED Rot', 'AOCS Roll', 'Jitter Yaw'], title='PSD TED directional')
plt.show()

# Save figure
#fig0.savefig('/home/nicholas/Nextcloud/presentations/presentation_PW12/plotPrimePSDYaw.png', bbox_inches='tight', dpi=300)
#fig1.savefig('/home/nicholas/Nextcloud/presentations/presentation_PW12/plotPrimePSDPitch.png', bbox_inches='tight', dpi=300)
#fig2.savefig('/home/nicholas/Nextcloud/presentations/presentation_PW12/plotPrimePSDRoll.png', bbox_inches='tight', dpi=300)

NameError: name 'time_ted' is not defined

## Prime AOCS and TED time series with sampling = 12.5 s and duration = 27 d

In [ ]:
filename = '/home/nicholas/software/python/platonium/models/aocs_prime_2021-01/03_PLATO_PDR_AOCSandTED.csv'
data = np.loadtxt(filename, delimiter=';')

time = data[:,0]  # [s]
x    = data[:,1]  # [arcsec]
y    = data[:,2]  # [arcsec]
z    = data[:,3]  # [arcsec]
data = [x, y, z]

In [ ]:
# Plot time series of yaw, pitch, and roll

fig = plt.figure(figsize=(8,9))
axes = plotYawPitchRollTimeSeries(fig, time/3600., data, units=['hours', 'arcsec'], title='Jitter+TED from Prime')
plt.show()

In [ ]:
# Plot PSD of yaw, pitch, and roll for Red Noise jitter

fig = plt.figure(figsize=(8,9))
plotYawPitchRollPSD(fig, time, data, title='PSD jitter + TED from Prime', xmin=1e2, misreq=True)
plt.show()